# 02 — Process European Commission legacy documents

## Purpose

This notebook converts the raw Commission documents inventoried by
`01_download_com_legacy2.ipynb` into reusable text representations.

The processing logic is intentionally simple enough to describe directly in a
Scientific Data Methods section:

1. verify the file-level handoff from notebook 01;
2. extract HTML text directly;
3. for PDFs, try PyMuPDF first;
4. if the native PDF text looks technically suspicious, try `pdftotext -layout`;
5. if the fallback text is still suspicious, run MinerU OCR;
6. retain the first extraction that passes the predefined technical checks;
7. write one canonical UTF-8 text file and one convenience Markdown file.

The notebook does not rediscover URLs, change document identifiers, classify cases,
or infer missing legal metadata. Those tasks belong elsewhere in the pipeline.

Detailed candidate scoring from earlier versions has been removed. The decision to
use a fallback method is based only on transparent technical conditions described below.


## Inputs and outputs

Notebook 02 uses the two concise tables produced by notebook 01:

- `output/com_legacy/com_legacy_documents.csv` — one row per retained raw document;
- `output/com_legacy/com_legacy_records.csv` — source-record metadata such as title,
  decision date and Commission case numbers.

All paths in the processing manifest are repository-relative.

For each successfully processed document the notebook creates:

- `data/processed/com_legacy/text/<document_id>.txt` — canonical lightly cleaned UTF-8 text;
- `data/processed/com_legacy/markdown/<document_id>.md` — a convenience Markdown
  representation with a small provenance header.

The canonical scientific representation is the `.txt` file. Markdown is provided
for easier human inspection and downstream reuse.


In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF
import pandas as pd
from bs4 import BeautifulSoup, UnicodeDammit
from charset_normalizer import from_bytes as charset_from_bytes
from tqdm import tqdm


def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "output").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. Run the notebook from inside the "
        "repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

INPUT_DOCUMENTS_CSV = PROJECT_ROOT / "output" / "com_legacy" / "com_legacy_documents.csv"
INPUT_RECORDS_CSV = PROJECT_ROOT / "output" / "com_legacy" / "com_legacy_records.csv"

PROCESSED_TEXT_DIR = PROJECT_ROOT / "data" / "processed" / "com_legacy" / "text"
PROCESSED_MARKDOWN_DIR = PROJECT_ROOT / "data" / "processed" / "com_legacy" / "markdown"
OUTPUT_DIR = PROJECT_ROOT / "output" / "com_legacy"
LOG_DIR = PROJECT_ROOT / "logs" / "com_legacy"

MANIFEST_CSV = OUTPUT_DIR / "com_legacy_clean_file_manifest.csv"
MANIFEST_XLSX = OUTPUT_DIR / "com_legacy_clean_file_manifest.xlsx"
PROCESSING_STATE_JSONL = LOG_DIR / "com_legacy_processing_state.jsonl"

for directory in [
    PROCESSED_TEXT_DIR,
    PROCESSED_MARKDOWN_DIR,
    OUTPUT_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Processing configuration
# ---------------------------------------------------------------------

PROCESSING_VERSION = "2.0"
FORCE_REPROCESS = False
VERIFY_RAW_SHA256 = True

RUN_PDFTOTEXT = True
RUN_OCR = True
OCR_TIMEOUT_SECONDS = 300

# Simple technical checks used to decide whether a fallback extractor is needed.
MIN_TOTAL_CHARS = 200
MIN_PDF_CHARS_PER_PAGE = 500
MAX_BAD_ENCODING_MARKERS = 3
MAX_ONE_CHAR_TOKEN_RATIO = 0.20

BAD_ENCODING_MARKERS = [
    "�",
    "Ã",
    "Â",
    "â€™",
    "â€œ",
    "â€",
    "ðŸ",
]

SUPPORTED_FILE_FORMATS = {"html", "pdf", "txt"}

PDFTOTEXT_CLI = shutil.which("pdftotext")
MINERU_CLI = shutil.which("mineru") or shutil.which("magic-pdf")


def repo_relative(path: Path) -> str:
    return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


print("Project root: .")
print("Input documents:", repo_relative(INPUT_DOCUMENTS_CSV))
print("Input records:", repo_relative(INPUT_RECORDS_CSV))
print("Processed text:", repo_relative(PROCESSED_TEXT_DIR))
print("Processed Markdown:", repo_relative(PROCESSED_MARKDOWN_DIR))
print("Processing state:", repo_relative(PROCESSING_STATE_JSONL))
print("pdftotext available:", PDFTOTEXT_CLI is not None)
print("MinerU available:", MINERU_CLI is not None)


Project root: .
Input documents: output/com_legacy/com_legacy_documents.csv
Input records: output/com_legacy/com_legacy_records.csv
Processed text: data/processed/com_legacy/text
Processed Markdown: data/processed/com_legacy/markdown
Processing state: logs/com_legacy/com_legacy_processing_state.jsonl
pdftotext available: True
MinerU available: True


## 1. Validate the handoff from notebook 01

The processing unit is the document row created by notebook 01. The existing
`document_id` is preserved unchanged.

Before extraction, the notebook checks only the things needed to ensure that it is
processing the intended raw files:

- document identifiers are present and unique;
- referenced source records exist;
- raw paths are repository-relative;
- raw files exist;
- SHA-256 hashes match the inventory from notebook 01.

A failure here stops processing rather than silently using a different file.


In [2]:
DOCUMENT_REQUIRED_COLUMNS = {
    "document_id",
    "source_entry_id",
    "related_source_entry_ids",
    "celex",
    "source_type",
    "language",
    "file_format",
    "source_url",
    "relative_path",
    "sha256",
}

RECORD_REQUIRED_COLUMNS = {
    "source_entry_id",
    "title",
    "decision_date",
    "case_numbers",
}


def read_csv_strings(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {repo_relative(path)}")
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def require_columns(frame: pd.DataFrame, required: set[str], label: str) -> None:
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def split_semicolon(value: str) -> list[str]:
    return [part.strip() for part in str(value or "").split(";") if part.strip()]


def portable_relative_path(value: str) -> bool:
    value = str(value or "").strip()
    if not value:
        return False
    path = Path(value)
    if path.is_absolute() or ".." in path.parts:
        return False
    if re.match(r"^[A-Za-z]:[\\/]", value):
        return False
    return True


documents_input = read_csv_strings(INPUT_DOCUMENTS_CSV)
records_input = read_csv_strings(INPUT_RECORDS_CSV)

require_columns(documents_input, DOCUMENT_REQUIRED_COLUMNS, "com_legacy_documents.csv")
require_columns(records_input, RECORD_REQUIRED_COLUMNS, "com_legacy_records.csv")

if documents_input["document_id"].eq("").any():
    raise ValueError("com_legacy_documents.csv contains an empty document_id.")

if not documents_input["document_id"].is_unique:
    duplicates = documents_input.loc[
        documents_input["document_id"].duplicated(keep=False),
        "document_id",
    ].tolist()
    raise ValueError(f"Duplicate document_id values: {duplicates[:10]}")

if records_input["source_entry_id"].eq("").any() or not records_input["source_entry_id"].is_unique:
    raise ValueError(
        "com_legacy_records.csv must contain unique, non-empty source_entry_id values."
    )

record_ids = set(records_input["source_entry_id"])
missing_primary = sorted(set(documents_input["source_entry_id"]) - record_ids)
if missing_primary:
    raise ValueError(
        "Document inventory references source_entry_id values absent from records: "
        f"{missing_primary[:10]}"
    )

related_missing: set[str] = set()
for value in documents_input["related_source_entry_ids"]:
    related_missing.update(set(split_semicolon(value)) - record_ids)

if related_missing:
    raise ValueError(
        "related_source_entry_ids contains identifiers absent from records: "
        f"{sorted(related_missing)[:10]}"
    )

bad_paths = documents_input.loc[
    ~documents_input["relative_path"].map(portable_relative_path),
    ["document_id", "relative_path"],
]
if not bad_paths.empty:
    raise ValueError(
        "Document inventory contains non-portable paths: "
        f"{bad_paths.head(10).to_dict('records')}"
    )

raw_paths = [PROJECT_ROOT / value for value in documents_input["relative_path"]]
missing_files = [
    documents_input.iloc[index]["document_id"]
    for index, path in enumerate(raw_paths)
    if not path.exists() or not path.is_file()
]
if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} inventoried raw files are missing; "
        f"examples: {missing_files[:10]}"
    )

bad_hash_format = ~documents_input["sha256"].str.fullmatch(r"[0-9a-fA-F]{64}")
if bad_hash_format.any():
    raise ValueError("The document inventory contains malformed SHA-256 values.")

if VERIFY_RAW_SHA256:
    actual_hashes = [
        sha256_file(path)
        for path in tqdm(
            raw_paths,
            desc="Verifying raw SHA-256",
            unit="file",
            dynamic_ncols=True,
        )
    ]
    expected_hashes = documents_input["sha256"].str.lower().tolist()
    mismatches = [
        documents_input.iloc[index]["document_id"]
        for index, (actual, expected) in enumerate(zip(actual_hashes, expected_hashes))
        if actual != expected
    ]
    if mismatches:
        raise ValueError(
            f"{len(mismatches)} raw files do not match notebook 01 SHA-256 values; "
            f"examples: {mismatches[:10]}"
        )

records_metadata = records_input[
    ["source_entry_id", "title", "decision_date", "case_numbers"]
].copy()

work_manifest = documents_input.merge(
    records_metadata,
    on="source_entry_id",
    how="left",
    validate="many_to_one",
)

work_manifest = work_manifest.rename(
    columns={
        "relative_path": "raw_path",
        "sha256": "raw_sha256",
    }
)
work_manifest["raw_sha256"] = work_manifest["raw_sha256"].str.lower()
work_manifest = work_manifest.sort_values("document_id").reset_index(drop=True)

print(f"Documents ready for processing: {len(work_manifest):,}")
print(f"Source records represented: {work_manifest['source_entry_id'].nunique():,}")
print("Raw-file integrity check: passed")
display(
    work_manifest[
        ["document_id", "source_entry_id", "file_format", "language", "raw_path"]
    ].head()
)


Verifying raw SHA-256: 100%|██████████| 1579/1579 [00:02<00:00, 734.23file/s]

Documents ready for processing: 1,579
Source records represented: 429
Raw-file integrity check: passed


,document_id,source_entry_id,file_format,language,raw_path
0,COMLEGDOC_00047bce38614bc8,COMLEG_1980_06da64ef1e1480da,html,DE,data/raw/com_legacy/files/html/31980D0917_DE.html
1,COMLEGDOC_0023b8b2a6c798e4,COMLEG_1986_bcc24ed8fb560aa1,pdf,DE,data/raw/com_legacy/files/pdfs/31987D0014_DE.pdf
2,COMLEGDOC_0035ad3bd3382123,COMLEG_1975_6185fe7c03945fb5,pdf,DE,data/raw/com_legacy/files/pdfs/31976D0159_DE.pdf
3,COMLEGDOC_00b8f642eee4c150,COMLEG_1983_750c0046fa9139a3,pdf,EN,data/raw/com_legacy/files/pdfs/31983D0400_EN.pdf
4,COMLEGDOC_00ca6d9a7111931f,COMLEG_1975_c95d76c15fabf61e,pdf,EN,data/raw/com_legacy/files/pdfs/31975D0649_EN.pdf


## 2. Text extraction strategy

### HTML

HTML files are decoded using their declared encoding where available, UTF-8 and
standard encoding-detection fallbacks. Navigation, scripts, forms and other obvious
site chrome are removed. The largest plausible content container is retained.

### PDF

PDF processing follows a fixed fallback hierarchy:

1. **PyMuPDF** sorted text is the default.
2. If that result is technically suspicious, **Poppler `pdftotext -layout`** is tried.
3. If the Poppler result is also suspicious, **MinerU OCR** is tried.
4. The first method that passes the technical checks is retained.
5. If no method passes, the best available non-empty fallback is still written but the
   document is marked `review`. If no method produces text at all, processing is marked
   `failed`.

This avoids comparing several methods with a bespoke weighted quality score.


## 3. When is extracted text considered suspicious?

The checks are deliberately simple and technical. A PDF extraction is sent to the
next fallback when any of the following applies:

- fewer than `MIN_TOTAL_CHARS` characters are extracted;
- the extracted text contains fewer than `MIN_PDF_CHARS_PER_PAGE` characters per page;
- at least `MAX_BAD_ENCODING_MARKERS` obvious encoding-corruption markers are present;
- more than `MAX_ONE_CHAR_TOKEN_RATIO` of alphabetic tokens contain only one character,
  which can indicate severely fragmented PDF text.

These checks are intended to identify extraction failures, not to assess the legal or
linguistic quality of a document. They do not use legal-word dictionaries, language
scores, duplicate-line scores or weighted quality metrics.

HTML and plain-text files use the same total-length, encoding-marker and fragmentation
checks for review purposes, but OCR is applied only to PDFs.


In [3]:
def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    replacements = {
        "\u00a0": " ",
        "\u200b": "",
        "\ufeff": "",
        "\r\n": "\n",
        "\r": "\n",
        "\f": "\n\n",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def clean_text_readable(text: str) -> str:
    """
    Light normalisation only.

    The canonical dataset keeps line and paragraph structure rather than flattening
    text specifically for regex or LLM use.
    """
    text = normalize_unicode(text)
    text = text.replace("\u00ad", "")

    # Join words split by line-break hyphenation.
    text = re.sub(
        r"(?<=[A-Za-zÀ-ÖØ-öø-ÿ])[-‐‑]\s*\n\s*(?=[a-zà-öø-ÿ])",
        "",
        text,
    )

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n" if text.strip() else ""


def count_bad_encoding_markers(text: str) -> int:
    return sum(text.count(marker) for marker in BAD_ENCODING_MARKERS)


def one_char_token_ratio(text: str) -> float:
    # Alphabetic tokens only; numbers and punctuation do not affect the ratio.
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", text or "")
    if not tokens:
        return 1.0
    return sum(len(token) == 1 for token in tokens) / len(tokens)


def assess_text(
    text: str,
    *,
    file_format: str,
    n_pages: int | None,
) -> dict[str, Any]:
    """
    Return transparent technical checks used by the fallback ladder.
    """
    text = text or ""
    n_chars = len(text)
    bad_marker_count = count_bad_encoding_markers(text)
    fragment_ratio = one_char_token_ratio(text)

    chars_per_page = None
    if file_format == "pdf" and n_pages:
        chars_per_page = n_chars / n_pages

    reasons: list[str] = []

    if n_chars < MIN_TOTAL_CHARS:
        reasons.append("too_little_text")

    if (
        file_format == "pdf"
        and chars_per_page is not None
        and chars_per_page < MIN_PDF_CHARS_PER_PAGE
    ):
        reasons.append("low_text_per_page")

    if bad_marker_count >= MAX_BAD_ENCODING_MARKERS:
        reasons.append("encoding_corruption")

    if fragment_ratio > MAX_ONE_CHAR_TOKEN_RATIO:
        reasons.append("fragmented_text")

    return {
        "passes": not reasons,
        "reasons": reasons,
        "n_chars": n_chars,
        "n_pages": n_pages,
        "chars_per_page": (
            round(chars_per_page, 1) if chars_per_page is not None else None
        ),
        "bad_encoding_markers": bad_marker_count,
        "one_char_token_ratio": round(fragment_ratio, 4),
    }


def decode_issue_count(text: str) -> int:
    control_chars = sum(
        1
        for char in text
        if unicodedata.category(char).startswith("C") and char not in "\n\t"
    )
    return count_bad_encoding_markers(text) + control_chars


def extract_declared_html_encoding(data: bytes) -> str | None:
    head = data[:8192].decode("ascii", errors="ignore")
    patterns = [
        r'<meta[^>]+charset\s*=\s*["\']?\s*([A-Za-z0-9._-]+)',
        r'<meta[^>]+content\s*=\s*["\'][^"\']*charset\s*=\s*([A-Za-z0-9._-]+)',
        r'<\?xml[^>]+encoding\s*=\s*["\']([^"\']+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, head, flags=re.I)
        if match:
            return match.group(1).strip()
    return None


def decode_bytes_smart(data: bytes, is_html: bool = False) -> tuple[str, dict[str, Any]]:
    """
    Try a small set of plausible encodings and choose the decoding with the fewest
    obvious corruption/control characters. Earlier candidates win ties.
    """
    candidates: list[tuple[str, str, str]] = []

    if data.startswith(b"\xef\xbb\xbf"):
        candidates.append(
            ("utf-8-sig", "bom", data.decode("utf-8-sig", errors="replace"))
        )
    elif data.startswith((b"\xff\xfe", b"\xfe\xff")):
        candidates.append(
            ("utf-16", "bom", data.decode("utf-16", errors="replace"))
        )

    declared = extract_declared_html_encoding(data) if is_html else None
    if declared:
        try:
            candidates.append(
                (declared, "declared", data.decode(declared, errors="replace"))
            )
        except (LookupError, UnicodeError):
            pass

    try:
        candidates.append(
            ("utf-8", "strict_utf8", data.decode("utf-8", errors="strict"))
        )
    except UnicodeDecodeError:
        pass

    if is_html:
        dammit = UnicodeDammit(data, is_html=True, smart_quotes_to=None)
        if dammit.unicode_markup:
            candidates.append(
                (
                    dammit.original_encoding or "unknown",
                    "unicode_dammit",
                    dammit.unicode_markup,
                )
            )

    try:
        for match in list(charset_from_bytes(data))[:3]:
            candidates.append(
                (
                    getattr(match, "encoding", None) or "unknown",
                    "charset_normalizer",
                    str(match),
                )
            )
    except (ValueError, TypeError):
        pass

    for encoding in ("cp1252", "latin-1"):
        candidates.append(
            (encoding, "fallback", data.decode(encoding, errors="replace"))
        )

    # Remove duplicate decoded strings while preserving candidate order.
    unique: list[tuple[str, str, str]] = []
    seen = set()
    for encoding, source, text in candidates:
        digest = hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()
        if digest not in seen:
            seen.add(digest)
            unique.append((encoding, source, text))

    encoding, source, text = min(
        unique,
        key=lambda item: decode_issue_count(item[2]),
    )

    return normalize_unicode(text), {
        "encoding": encoding,
        "encoding_source": source,
    }


## 4. HTML, PDF and OCR extractors

The extraction functions below do not decide which method is scientifically “best”.
They simply return text and basic metadata. The fallback decision is made separately
using the transparent checks above.

For HTML, the largest plausible cleaned content container is used. For PDFs, PyMuPDF
and `pdftotext` are native-text methods; MinerU is the OCR fallback.


In [4]:
HTML_REMOVE_SELECTORS = [
    "script",
    "style",
    "noscript",
    "nav",
    "footer",
    "header",
    "form",
    "aside",
    "[class*='cookie']",
    "[id*='cookie']",
    "[class*='share']",
    "[class*='language']",
    "[aria-label*='language']",
    "[class*='pagination']",
    "[class*='print']",
]

HTML_CANDIDATE_SELECTORS = [
    "main",
    "article",
    "#document1",
    "#TexteOnly",
    "#text",
    ".document-content",
    ".document",
    ".content",
    "body",
]


def extract_text_from_html(path: Path) -> tuple[str, dict[str, Any]]:
    html, decode_meta = decode_bytes_smart(path.read_bytes(), is_html=True)
    soup = BeautifulSoup(html, "html.parser")

    for selector in HTML_REMOVE_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()

    candidates: list[tuple[int, str, str]] = []
    for selector in HTML_CANDIDATE_SELECTORS:
        for tag in soup.select(selector):
            text = tag.get_text("\n", strip=True)
            if len(text) >= 100:
                candidates.append((len(text), selector, text))

    if candidates:
        _, selected_selector, text = max(candidates, key=lambda item: item[0])
    else:
        selected_selector = "document"
        text = soup.get_text("\n", strip=True)

    return normalize_unicode(text), {
        "method": "html",
        "n_pages": None,
        "html_selected_container": selected_selector,
        **decode_meta,
    }


def extract_text_from_plain(path: Path) -> tuple[str, dict[str, Any]]:
    text, decode_meta = decode_bytes_smart(path.read_bytes(), is_html=False)
    return text, {
        "method": "plain_text",
        "n_pages": None,
        **decode_meta,
    }


def extract_pdf_pymupdf(path: Path) -> tuple[str, dict[str, Any]]:
    page_text: list[str] = []
    with fitz.open(path) as document:
        for page in document:
            page_text.append(page.get_text("text", sort=True) or "")
        n_pages = len(document)

    return normalize_unicode("\n\n".join(page_text)), {
        "method": "pymupdf",
        "n_pages": n_pages,
    }


def extract_pdf_pdftotext(path: Path) -> tuple[str, dict[str, Any]]:
    if PDFTOTEXT_CLI is None:
        raise RuntimeError("pdftotext is not available on PATH.")

    with tempfile.TemporaryDirectory(prefix="pdftotext_") as tmpdir:
        output_path = Path(tmpdir) / "output.txt"
        command = [
            PDFTOTEXT_CLI,
            "-layout",
            "-enc",
            "UTF-8",
            str(path),
            str(output_path),
        ]
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=300,
            check=False,
        )
        if completed.returncode != 0:
            raise RuntimeError(completed.stderr[-2000:] or "pdftotext failed")
        text = output_path.read_text(encoding="utf-8", errors="replace")

    try:
        with fitz.open(path) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    return normalize_unicode(text), {
        "method": "pdftotext",
        "n_pages": n_pages,
    }


def run_mineru(input_pdf: Path, output_dir: Path) -> None:
    if MINERU_CLI is None:
        raise RuntimeError("MinerU is not available on PATH.")

    command = [
        MINERU_CLI,
        "-p",
        str(input_pdf),
        "-o",
        str(output_dir),
    ]
    environment = os.environ.copy()
    environment.setdefault("ORT_LOG_SEVERITY_LEVEL", "3")

    completed = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=OCR_TIMEOUT_SECONDS,
        env=environment,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(completed.stderr[-2000:] or "MinerU failed")


def find_mineru_text(output_dir: Path) -> Path | None:
    # Prefer Markdown when available because it is MinerU's richest text output.
    markdown = [
        path
        for path in output_dir.glob("**/*.md")
        if path.is_file() and path.stat().st_size > 0
    ]
    if markdown:
        return max(markdown, key=lambda path: path.stat().st_size)

    text_files = [
        path
        for path in output_dir.glob("**/*.txt")
        if path.is_file() and path.stat().st_size > 0
    ]
    if text_files:
        return max(text_files, key=lambda path: path.stat().st_size)

    return None


def extract_pdf_mineru(path: Path) -> tuple[str, dict[str, Any]]:
    with tempfile.TemporaryDirectory(prefix="mineru_") as tmpdir:
        output_dir = Path(tmpdir)
        run_mineru(path, output_dir)

        text_path = find_mineru_text(output_dir)
        if text_path is None:
            raise RuntimeError(
                "MinerU completed but produced no non-empty Markdown/text file."
            )

        text = text_path.read_text(encoding="utf-8", errors="replace")
        mineru_source_format = text_path.suffix.lower().lstrip(".")

    try:
        with fitz.open(path) as document:
            n_pages = len(document)
    except Exception:
        n_pages = None

    return normalize_unicode(text), {
        "method": "mineru_ocr",
        "n_pages": n_pages,
        "mineru_source_format": mineru_source_format,
    }


def try_extractor(
    path: Path,
    file_format: str,
    extractor,
) -> tuple[dict[str, Any] | None, str]:
    try:
        raw_text, meta = extractor(path)
        clean_text = clean_text_readable(raw_text)
        assessment = assess_text(
            clean_text,
            file_format=file_format,
            n_pages=meta.get("n_pages"),
        )
        return {
            "method": meta.get("method", ""),
            "text": clean_text,
            "meta": meta,
            "assessment": assessment,
        }, ""
    except Exception as exc:
        return None, repr(exc)


## 5. PDF fallback decision

The fallback sequence is deterministic.

A method is accepted as soon as its output passes the four technical checks. This
means OCR is neither run for every PDF nor reserved only for completely empty scans:
it is used when both native methods produce output that remains technically suspicious.

If all available methods remain suspicious, processing does not silently discard the
document. The last non-empty result is retained and the row is marked for manual review.


In [5]:
def process_pdf(path: Path) -> dict[str, Any]:
    attempts: list[dict[str, Any]] = []
    errors: list[str] = []
    fallback_reason: list[str] = []

    # 1. Default native extraction
    pymupdf_result, error = try_extractor(path, "pdf", extract_pdf_pymupdf)
    if error:
        errors.append(f"pymupdf: {error}")
        fallback_reason.append("pymupdf_failed")
    elif pymupdf_result is not None:
        attempts.append(pymupdf_result)
        if pymupdf_result["assessment"]["passes"]:
            return {
                "selected": pymupdf_result,
                "status": "success",
                "review_reason": "",
                "fallback_reason": "",
                "ocr_used": False,
                "errors": errors,
            }
        fallback_reason.extend(
            f"pymupdf:{reason}"
            for reason in pymupdf_result["assessment"]["reasons"]
        )

    # 2. Cheap native fallback
    if RUN_PDFTOTEXT and PDFTOTEXT_CLI is not None:
        poppler_result, error = try_extractor(
            path,
            "pdf",
            extract_pdf_pdftotext,
        )
        if error:
            errors.append(f"pdftotext: {error}")
            fallback_reason.append("pdftotext_failed")
        elif poppler_result is not None:
            attempts.append(poppler_result)
            if poppler_result["assessment"]["passes"]:
                return {
                    "selected": poppler_result,
                    "status": "success",
                    "review_reason": "",
                    "fallback_reason": "; ".join(fallback_reason),
                    "ocr_used": False,
                    "errors": errors,
                }
            fallback_reason.extend(
                f"pdftotext:{reason}"
                for reason in poppler_result["assessment"]["reasons"]
            )
    elif RUN_PDFTOTEXT:
        fallback_reason.append("pdftotext_unavailable")

    # 3. OCR fallback
    if RUN_OCR and MINERU_CLI is not None:
        ocr_result, error = try_extractor(
            path,
            "pdf",
            extract_pdf_mineru,
        )
        if error:
            errors.append(f"mineru_ocr: {error}")
            fallback_reason.append("mineru_failed")
        elif ocr_result is not None:
            attempts.append(ocr_result)
            if ocr_result["assessment"]["passes"]:
                return {
                    "selected": ocr_result,
                    "status": "success",
                    "review_reason": "",
                    "fallback_reason": "; ".join(fallback_reason),
                    "ocr_used": True,
                    "errors": errors,
                }

            # OCR returned text but it still failed one or more technical checks.
            return {
                "selected": ocr_result,
                "status": "review",
                "review_reason": "; ".join(
                    f"mineru:{reason}"
                    for reason in ocr_result["assessment"]["reasons"]
                ),
                "fallback_reason": "; ".join(fallback_reason),
                "ocr_used": True,
                "errors": errors,
            }
    elif RUN_OCR:
        fallback_reason.append("mineru_unavailable")

    # No method passed. Retain the latest non-empty result for review.
    nonempty_attempts = [
        attempt for attempt in attempts if attempt.get("text", "").strip()
    ]
    if nonempty_attempts:
        selected = nonempty_attempts[-1]
        return {
            "selected": selected,
            "status": "review",
            "review_reason": "; ".join(
                f"{selected['method']}:{reason}"
                for reason in selected["assessment"]["reasons"]
            ) or "no_method_passed_checks",
            "fallback_reason": "; ".join(fallback_reason),
            "ocr_used": selected["method"] == "mineru_ocr",
            "errors": errors,
        }

    return {
        "selected": None,
        "status": "failed",
        "review_reason": "no_text_extracted",
        "fallback_reason": "; ".join(fallback_reason),
        "ocr_used": False,
        "errors": errors,
    }


def process_non_pdf(path: Path, file_format: str) -> dict[str, Any]:
    extractor = (
        extract_text_from_html
        if file_format == "html"
        else extract_text_from_plain
    )
    result, error = try_extractor(path, file_format, extractor)

    if result is None:
        return {
            "selected": None,
            "status": "failed",
            "review_reason": "extraction_failed",
            "fallback_reason": "",
            "ocr_used": False,
            "errors": [error] if error else [],
        }

    if result["assessment"]["passes"]:
        status = "success"
        review_reason = ""
    else:
        status = "review"
        review_reason = "; ".join(result["assessment"]["reasons"])

    return {
        "selected": result,
        "status": status,
        "review_reason": review_reason,
        "fallback_reason": "",
        "ocr_used": False,
        "errors": [error] if error else [],
    }


## 6. Canonical text and Markdown representations

Selected text receives only light normalization:

- Unicode NFKC normalization;
- removal of byte-order marks, zero-width spaces and soft hyphens;
- normalization of ordinary whitespace and line endings;
- joining of words broken across line-ending hyphens;
- preservation of line and paragraph structure.

The canonical text is not flattened specifically for regex or LLM use. Task-specific
transformations can be created downstream without changing the deposited processed data.

The Markdown file contains a small provenance header followed by the same canonical
text. It is a convenience representation rather than a second independently cleaned
version of the document.


In [6]:
def yaml_string(value: Any) -> str:
    # JSON string syntax is valid for quoted YAML scalar values.
    return json.dumps("" if value is None else str(value), ensure_ascii=False)


def write_markdown(
    row: dict[str, Any],
    text: str,
    path: Path,
    *,
    extraction_method: str,
    processing_status: str,
) -> None:
    lines = [
        "---",
        f"document_id: {yaml_string(row.get('document_id'))}",
        f"source_entry_id: {yaml_string(row.get('source_entry_id'))}",
        f"title: {yaml_string(row.get('title'))}",
        f"decision_date: {yaml_string(row.get('decision_date'))}",
        f"case_numbers: {yaml_string(row.get('case_numbers'))}",
        f"celex: {yaml_string(row.get('celex'))}",
        f"language: {yaml_string(row.get('language'))}",
        f"source_url: {yaml_string(row.get('source_url'))}",
        f"extraction_method: {yaml_string(extraction_method)}",
        f"processing_status: {yaml_string(processing_status)}",
        "---",
        "",
        text.rstrip(),
        "",
    ]
    path.write_text("\n".join(lines), encoding="utf-8")


## 7. Resumable processing state

`logs/com_legacy/com_legacy_processing_state.jsonl` is an operational checkpoint,
not a publication data table.

After each document is processed, the current state is written atomically. A successful
or review-status document can be reused on a later run when:

- the raw SHA-256 is unchanged;
- the processing version is unchanged;
- the text and Markdown outputs still exist;
- the stored text SHA-256 matches the current file.

Failed documents are retried automatically. Set `FORCE_REPROCESS=True` to rebuild all
documents deliberately.

The checkpoint stores only information useful for resuming or troubleshooting; it does
not retain candidate-score matrices or dozens of diagnostic ratios.


In [7]:
def json_safe(value: Any) -> Any:
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        if isinstance(value, float) and pd.isna(value):
            return None
        return value
    if isinstance(value, Path):
        return value.as_posix()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if pd.isna(value):
        return None
    return str(value)


def load_processing_state() -> dict[str, dict[str, Any]]:
    if not PROCESSING_STATE_JSONL.exists():
        return {}

    state: dict[str, dict[str, Any]] = {}
    with PROCESSING_STATE_JSONL.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Malformed JSONL state at line {line_number}: {exc}"
                ) from exc

            document_id = str(record.get("document_id", "")).strip()
            if not document_id:
                raise ValueError(
                    f"State line {line_number} has no document_id."
                )
            state[document_id] = record

    return state


def save_processing_state(state: dict[str, dict[str, Any]]) -> None:
    temp_path = PROCESSING_STATE_JSONL.with_suffix(".jsonl.tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        for document_id in sorted(state):
            handle.write(
                json.dumps(
                    json_safe(state[document_id]),
                    ensure_ascii=False,
                    sort_keys=True,
                )
                + "\n"
            )
    os.replace(temp_path, PROCESSING_STATE_JSONL)


def expected_text_path(document_id: str) -> Path:
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", str(document_id)):
        raise ValueError(f"document_id is not safe for a filename: {document_id}")
    return PROCESSED_TEXT_DIR / f"{document_id}.txt"


def expected_markdown_path(document_id: str) -> Path:
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", str(document_id)):
        raise ValueError(f"document_id is not safe for a filename: {document_id}")
    return PROCESSED_MARKDOWN_DIR / f"{document_id}.md"


def state_is_reusable(
    state_record: dict[str, Any] | None,
    input_row: dict[str, Any],
) -> bool:
    if FORCE_REPROCESS or not state_record:
        return False

    if state_record.get("processing_status") not in {"success", "review"}:
        return False

    if state_record.get("processing_version") != PROCESSING_VERSION:
        return False

    if str(state_record.get("raw_sha256", "")).lower() != str(
        input_row["raw_sha256"]
    ).lower():
        return False

    text_rel = str(state_record.get("text_path", "")).strip()
    markdown_rel = str(state_record.get("markdown_path", "")).strip()
    text_sha256 = str(state_record.get("text_sha256", "")).lower()

    if not text_rel or not markdown_rel:
        return False

    if not portable_relative_path(text_rel) or not portable_relative_path(markdown_rel):
        return False

    text_path = PROJECT_ROOT / text_rel
    markdown_path = PROJECT_ROOT / markdown_rel

    if not text_path.exists() or text_path.stat().st_size <= 0:
        return False
    if not markdown_path.exists() or markdown_path.stat().st_size <= 0:
        return False

    if not re.fullmatch(r"[0-9a-f]{64}", text_sha256):
        return False

    return sha256_file(text_path) == text_sha256


def process_document(input_row: dict[str, Any]) -> dict[str, Any]:
    document_id = str(input_row["document_id"])
    file_format = str(input_row["file_format"]).strip().lower()
    raw_path = PROJECT_ROOT / str(input_row["raw_path"])

    text_path = expected_text_path(document_id)
    markdown_path = expected_markdown_path(document_id)

    base = {
        "document_id": document_id,
        "processing_version": PROCESSING_VERSION,
        "raw_sha256": str(input_row["raw_sha256"]).lower(),
        "updated_at_utc": utc_now_iso(),
    }

    if file_format not in SUPPORTED_FILE_FORMATS:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": "",
            "review_reason": "unsupported_file_format",
            "text_path": "",
            "markdown_path": "",
            "text_sha256": "",
            "n_chars": 0,
            "n_pages": None,
            "chars_per_page": None,
            "error": f"unsupported_file_format:{file_format}",
        }

    result = (
        process_pdf(raw_path)
        if file_format == "pdf"
        else process_non_pdf(raw_path, file_format)
    )

    selected = result["selected"]

    if selected is None:
        return {
            **base,
            "processing_status": "failed",
            "extraction_method": "",
            "ocr_used": False,
            "fallback_reason": result.get("fallback_reason", ""),
            "review_reason": result.get("review_reason", "no_text_extracted"),
            "text_path": "",
            "markdown_path": "",
            "text_sha256": "",
            "n_chars": 0,
            "n_pages": None,
            "chars_per_page": None,
            "error": " | ".join(result.get("errors", [])) or "no_text_extracted",
        }

    text = selected["text"]
    text_path.write_text(text, encoding="utf-8")

    write_markdown(
        input_row,
        text,
        markdown_path,
        extraction_method=selected["method"],
        processing_status=result["status"],
    )

    assessment = selected["assessment"]

    return {
        **base,
        "processing_status": result["status"],
        "extraction_method": selected["method"],
        "ocr_used": bool(result["ocr_used"]),
        "fallback_reason": result.get("fallback_reason", ""),
        "review_reason": result.get("review_reason", ""),
        "text_path": repo_relative(text_path),
        "markdown_path": repo_relative(markdown_path),
        "text_sha256": sha256_file(text_path),
        "n_chars": assessment["n_chars"],
        "n_pages": assessment["n_pages"],
        "chars_per_page": assessment["chars_per_page"],
        "error": " | ".join(result.get("errors", [])),
    }


## 8. Process documents

The progress bar reports successful extractions, review cases, failures, OCR use and
checkpoint reuse.

A `review` status means text was retained but one or more minimum technical checks
still failed after the available fallback sequence. These rows are the natural focus
for manual inspection.


In [8]:
processing_state = load_processing_state()
manifest_rows: list[dict[str, Any]] = []

run_counts = {
    "success": 0,
    "review": 0,
    "failed": 0,
    "ocr": 0,
    "reused": 0,
}

target_rows = work_manifest.to_dict("records")
progress = tqdm(
    target_rows,
    desc="Processing COM legacy documents",
    unit="doc",
    dynamic_ncols=True,
)

for input_row in progress:
    document_id = str(input_row["document_id"])
    previous_state = processing_state.get(document_id)

    if state_is_reusable(previous_state, input_row):
        state_record = previous_state
        run_counts["reused"] += 1
    else:
        state_record = process_document(input_row)
        processing_state[document_id] = state_record
        save_processing_state(processing_state)

    manifest_row = {
        "document_id": input_row["document_id"],
        "source_entry_id": input_row["source_entry_id"],
        "title": input_row["title"],
        "decision_date": input_row["decision_date"],
        "case_numbers": input_row["case_numbers"],
        "celex": input_row["celex"],
        "language": input_row["language"],
        "file_format": input_row["file_format"],
        "source_url": input_row["source_url"],
        "raw_path": input_row["raw_path"],
        "text_path": state_record.get("text_path", ""),
        "markdown_path": state_record.get("markdown_path", ""),
        "extraction_method": state_record.get("extraction_method", ""),
        "ocr_used": state_record.get("ocr_used", False),
        "processing_status": state_record.get("processing_status", ""),
        "review_reason": state_record.get("review_reason", ""),
        "fallback_reason": state_record.get("fallback_reason", ""),
        "raw_sha256": input_row["raw_sha256"],
        "text_sha256": state_record.get("text_sha256", ""),
        "n_pages": state_record.get("n_pages"),
        "n_chars": state_record.get("n_chars"),
        "chars_per_page": state_record.get("chars_per_page"),
        "error": state_record.get("error", ""),
    }
    manifest_rows.append(manifest_row)

    status = state_record.get("processing_status", "failed")
    if status in run_counts:
        run_counts[status] += 1
    else:
        run_counts["failed"] += 1

    if state_record.get("ocr_used"):
        run_counts["ocr"] += 1

    progress.set_postfix(
        {
            "ok": run_counts["success"],
            "review": run_counts["review"],
            "fail": run_counts["failed"],
            "ocr": run_counts["ocr"],
            "reused": run_counts["reused"],
        },
        refresh=False,
    )

processing_manifest = (
    pd.DataFrame(manifest_rows)
    .sort_values("document_id")
    .reset_index(drop=True)
)

print(f"Documents considered: {len(processing_manifest):,}")
print(f"Successful: {(processing_manifest['processing_status'] == 'success').sum():,}")
print(f"Review: {(processing_manifest['processing_status'] == 'review').sum():,}")
print(f"Failed: {(processing_manifest['processing_status'] == 'failed').sum():,}")
print(f"OCR selected: {processing_manifest['ocr_used'].fillna(False).astype(bool).sum():,}")
print(f"Checkpoints reused this run: {run_counts['reused']:,}")


Processing COM legacy documents: 100%|██████████| 1579/1579 [00:00<00:00, 2757.99doc/s, ok=1577, review=2, fail=0, ocr=1, reused=1579]

Documents considered: 1,579
Successful: 1,577
Review: 2
Failed: 0
OCR selected: 1
Checkpoints reused this run: 1,579


## 9. Publication/review outputs

The processing manifest is deliberately compact. It records source provenance, raw and
processed paths, the selected extraction method, whether OCR was used, processing
status and the minimum information needed to inspect exceptional cases.

The Excel workbook contains only three sheets:

- `documents` — the complete compact processing manifest;
- `review` — rows with `review` or `failed` status;
- `summary` — method counts, status counts and the fixed processing thresholds.

There is no separate validation CSV or validation sheet. Hard integrity failures stop
the notebook directly; review-worthy extraction cases are shown explicitly in the
`review` sheet.


In [9]:
processing_manifest.to_csv(
    MANIFEST_CSV,
    index=False,
    encoding="utf-8",
)

review_df = processing_manifest.loc[
    processing_manifest["processing_status"].isin(["review", "failed"])
].copy()

status_counts = (
    processing_manifest["processing_status"]
    .value_counts(dropna=False)
    .rename_axis("value")
    .reset_index(name="documents")
)
status_counts.insert(0, "category", "processing_status")

method_counts = (
    processing_manifest["extraction_method"]
    .replace("", "(none)")
    .value_counts(dropna=False)
    .rename_axis("value")
    .reset_index(name="documents")
)
method_counts.insert(0, "category", "extraction_method")

summary_rows = pd.DataFrame(
    [
        ["configuration", "processing_version", PROCESSING_VERSION],
        ["configuration", "min_total_chars", MIN_TOTAL_CHARS],
        ["configuration", "min_pdf_chars_per_page", MIN_PDF_CHARS_PER_PAGE],
        ["configuration", "max_bad_encoding_markers", MAX_BAD_ENCODING_MARKERS],
        ["configuration", "max_one_char_token_ratio", MAX_ONE_CHAR_TOKEN_RATIO],
        ["configuration", "run_pdftotext", RUN_PDFTOTEXT],
        ["configuration", "run_ocr", RUN_OCR],
        ["software", "pandas", importlib.metadata.version("pandas")],
        ["software", "PyMuPDF", importlib.metadata.version("PyMuPDF")],
        ["software", "beautifulsoup4", importlib.metadata.version("beautifulsoup4")],
        ["software", "charset-normalizer", importlib.metadata.version("charset-normalizer")],
    ],
    columns=["category", "value", "documents"],
)

processing_summary = pd.concat(
    [status_counts, method_counts, summary_rows],
    ignore_index=True,
)

with pd.ExcelWriter(MANIFEST_XLSX, engine="openpyxl") as writer:
    processing_manifest.to_excel(
        writer,
        sheet_name="documents",
        index=False,
    )
    review_df.to_excel(
        writer,
        sheet_name="review",
        index=False,
    )
    processing_summary.to_excel(
        writer,
        sheet_name="summary",
        index=False,
    )

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font

wb = load_workbook(MANIFEST_XLSX)

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    if ws.max_row >= 1 and ws.max_column >= 1:
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(bold=True)

    for column_cells in ws.columns:
        letter = column_cells[0].column_letter
        values = [
            str(cell.value or "")
            for cell in column_cells[: min(len(column_cells), 150)]
        ]
        width = min(
            max(max((len(value) for value in values), default=8) + 2, 10),
            50,
        )
        ws.column_dimensions[letter].width = width

    for row in ws.iter_rows():
        for cell in row:
            if isinstance(cell.value, str) and len(cell.value) > 80:
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True,
                )

wb.save(MANIFEST_XLSX)

print("Outputs:")
print("-", repo_relative(MANIFEST_CSV))
print("-", repo_relative(MANIFEST_XLSX))
print("-", repo_relative(PROCESSING_STATE_JSONL))
print("-", repo_relative(PROCESSED_TEXT_DIR) + "/<document_id>.txt")
print("-", repo_relative(PROCESSED_MARKDOWN_DIR) + "/<document_id>.md")

if len(review_df):
    print()
    print(f"Rows requiring review: {len(review_df):,}")
    display(
        review_df[
            [
                "document_id",
                "title",
                "file_format",
                "extraction_method",
                "processing_status",
                "review_reason",
                "fallback_reason",
                "error",
            ]
        ].head(25)
    )


Outputs:
- output/com_legacy/com_legacy_clean_file_manifest.csv
- output/com_legacy/com_legacy_clean_file_manifest.xlsx
- logs/com_legacy/com_legacy_processing_state.jsonl
- data/processed/com_legacy/text/<document_id>.txt
- data/processed/com_legacy/markdown/<document_id>.md

Rows requiring review: 2


,document_id,title,file_format,extraction_method,processing_status,review_reason,fallback_reason,error
666,COMLEGDOC_6cfb18963e6ce1de,TACA,pdf,pdftotext,review,pdftotext:encoding_corruption,pymupdf:encoding_corruption; pdftotext:encodin...,mineru_ocr: TimeoutExpired(['/home/edik/projec...
751,COMLEGDOC_798c3e349bc21371,TACA,pdf,pdftotext,review,pdftotext:encoding_corruption,pymupdf:encoding_corruption; pdftotext:encodin...,mineru_ocr: TimeoutExpired(['/home/edik/projec...


## 10. Technical validation for the Data Descriptor

The automated checks in this notebook are intentionally limited to file integrity and
obvious extraction failures. They are not presented as a substitute for human
validation.

For the Scientific Data release, a practical validation strategy is:

- inspect every document for which OCR was selected;
- inspect every row marked `review` or `failed`;
- inspect a stratified sample of ordinary successful extractions across decades and
  source formats.

Manual inspection should confirm that the processed text corresponds to the source,
that substantial text has not been omitted, that obvious encoding corruption is absent,
and that the document remains broadly readable.

The final sample size and sampling rule should be fixed and reported in the paper once
the full dataset has been processed.


## Reproducibility notes

Required Python packages are `pandas`, `tqdm`, `beautifulsoup4`,
`charset-normalizer`, `PyMuPDF` and `openpyxl`.

Poppler's `pdftotext` is the preferred native fallback for suspicious PDFs. MinerU is
the OCR fallback. Their availability is reported when the notebook starts.

The raw documents remain unchanged. The processed `.txt` and `.md` files can therefore
always be recreated from the raw inventory and this notebook. Repository-relative paths
make the manifest portable across machines and suitable for a frozen GitHub/Zenodo
release.
